# Kaggle 免费 GPU 跑 Qwen2.5-Coder-14B（写代码）

目标：Kaggle T4 x2（每张 16GB）上用 Ollama 跑 `qwen2.5-coder:14b`（Q4_K_M 约 9GB，单卡够用），先在 notebook 里验证，再用 VS Code Tunnel + Continue 插件在本地 VS Code 里当代码助手。

前置条件（新建 notebook 时就要选好）：
- Accelerator 选 **GPU T4 x2**；右侧 Settings 里 **Internet 必须 ON**（否则拉不到 Ollama 和模型）。
- 免费额度约 30 小时/周，跑的是 GPU 会话时长；注意右上角会话计时，别挂着空跑。
- 默认按你的主业准备：C/C++、Qt/QML；这个模型写 Python 同样适用，换提示词即可，不用换模型。
- 本 notebook 未在 Kaggle 上实测过：前面先是检查 cell（GPU、磁盘、联网），失败会直接报错停在那一步，按报错修，不会悄悄跳过。

In [ ]:
# 1. 先验环境：GPU、磁盘、联网（任一项不对就先停）
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1
!curl -sI --max-time 15 https://ollama.com | head -1

In [ ]:
# 2. 装 Ollama（Kaggle notebook 是 root 身份，install.sh 一般直接能装）
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# 3. 启动 Ollama 服务（模型放 /kaggle/working，写得下 9GB；先看上一步 df 的剩余空间）
import os, subprocess, time, urllib.request
os.makedirs('/kaggle/working/ollama-models', exist_ok=True)
env = dict(os.environ, OLLAMA_MODELS='/kaggle/working/ollama-models')
log = open('/kaggle/working/ollama.log', 'wb')
subprocess.Popen(['ollama', 'serve'], env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/', timeout=2)
        print('ollama serve is up')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ollama serve 未起来，看 /kaggle/working/ollama.log')

In [ ]:
# 4. 拉模型：qwen2.5-coder:14b，Q4_K_M 约 9.0GB，单张 T4 能装下，还剩约 7GB 给 KV cache
#    第一次拉要几分钟，看进度条；中途别关会话。
!OLLAMA_MODELS=/kaggle/working/ollama-models ollama pull qwen2.5-coder:14b

In [ ]:
# 5. 定上下文 16k：默认上下文偏小，长文件喂不进去；16k 下 14B 在 16GB 单卡上仍有余量
open('/kaggle/working/Modelfile', 'w').write('FROM qwen2.5-coder:14b\nPARAMETER num_ctx 16384\n')
!OLLAMA_MODELS=/kaggle/working/ollama-models ollama create qwen-coder-14b -f /kaggle/working/Modelfile
!OLLAMA_MODELS=/kaggle/working/ollama-models ollama list

In [ ]:
# 6. 冒烟测试：一段 C++ 代码补全，确认模型真在 GPU 上跑、能出代码
import json, urllib.request
req = urllib.request.Request('http://localhost:11434/api/chat', data=json.dumps({
    'model': 'qwen-coder-14b', 'stream': False,
    'messages': [{'role': 'user', 'content': '用 C++17 写一个函数：输入 std::vector<int>，返回其中出现次数最多的元素，写清时间复杂度，只给代码加一行注释说明思路。'}],
}).encode(), headers={'Content-Type': 'application/json'})
print(json.loads(urllib.request.urlopen(req, timeout=300).read())['message']['content'])

## 7. 接入本地 VS Code（可选，跑通上面再做）

Kaggle 不直接对外暴露端口，走 VS Code Tunnel（出站联网，不用开端口）：

```bash
# 在 notebook 的 terminal 里（或加一个 ! 开头的 cell）
curl -Lk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' --output /kaggle/working/vscode_cli.tar.gz
tar -xf /kaggle/working/vscode_cli.tar.gz -C /kaggle/working/
/kaggle/working/code tunnel --accept-server-license-terms
```

按提示在你自己电脑的浏览器里完成 GitHub/Microsoft 登录（设备码授权，别把码贴给别人）。
然后本地 VS Code 左下角 → “连接到隧道” → 选这个隧道；隧道里的 VS Code 装 **Continue** 插件，配置见同目录 `continue-config.yaml`（模型指向隧道内的 `http://localhost:11434`，因为 Continue 和 Ollama 跑在同一个 Kaggle 会话里）。

## 注意事项

- 模型只在当前会话有效：Kaggle 会话结束，`/kaggle/working` 下的东西不保留，下次要重新拉（9GB）。想省时间就一次把活干完。
- 别贪大：`qwen2.5-coder:32b` / `qwen3:32b` 约 19–20GB，单张 T4 装不下；双卡要走 vLLM 拆分，那是另一套写法，不在本 notebook 里。
- Devstral Small 2 24B（agent 多步改代码更强）可以同法换模型名拉，但 Q4_K_M 约 15GB、只剩约 1GB 给 KV cache，要把 num_ctx 改小才稳。
- 速度预期：T4 上 14B 约 15–30 tok/s，够写代码、改 Bug、写单测，别跟本地 4090 比。